In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import requests
from io import StringIO

pd.options.mode.copy_on_write = True

# Bygninger og Energiforbruk - Årlig oppløsning per byggeår

Henter data fra SSB for **2022**:
- **06266**: Antall eneboliger etter byggeår
- **10576**: Gjennomsnittlig energiforbruk per husholdning etter energibærer og byggeår

Splitter begge datasett til årlig oppløsning:
- **Antall boliger** (sum) → fordeles jevnt over årene i intervallet
- **Energiforbruk** (gjennomsnitt) → kopieres til hvert år i intervallet

In [2]:
# Hent antall eneboliger per byggeår fra SSB 06266
r = requests.get("https://data.ssb.no/api/pxwebapi/v2/tables/06266/data", params={
    'lang': 'no',
    'outputformat': 'csv',
    'valueCodes[Region]': '0',
    'valueCodes[BygnType]': '01',
    'valueCodes[BygnAr]': '*',
    'valueCodes[ContentsCode]': 'Boliger',
    'valueCodes[Tid]': '2022',
})
df_buildings = pd.read_csv(StringIO(r.text)).rename(columns={'Boliger 2022': 'Boliger'})
df_buildings

,Region,BygnType,BygnAr,Boliger
0,0,1,1,79505
1,0,1,2,48140
2,0,1,3,75266
3,0,1,4,8137
4,0,1,6,169835
5,0,1,7,158873
6,0,1,8,215772
7,0,1,9,195250
8,0,1,10,105851
9,0,1,11,93568


In [3]:

# Definer årsintervaller for 06266 byggeår-koder
bygn_year_ranges = {
    1:  (1900, 1900),  # 1900 og tidligere
    2:  (1901, 1920),
    3:  (1921, 1940),
    4:  (1941, 1945),
    6:  (1946, 1960),
    7:  (1961, 1970),
    8:  (1971, 1980),
    9:  (1981, 1990),
    10: (1991, 2000),
    11: (2001, 2010),
    12: (2011, 2020),
    13: (2021, 2022),
    99: (1970, 1970),  # Ukjent byggeår → tilordnet 1970
}

expand_bygn = pd.DataFrame([
    {'BygnAr': code, 'byggeaar': y, 'n_years': end - start + 1}
    for code, (start, end) in bygn_year_ranges.items()
    for y in range(start, end + 1)
])

bygningsbestand_2022 = (
    df_buildings
    .merge(expand_bygn, on='BygnAr')
    .assign(antall_boliger=lambda d: d['Boliger'] / d['n_years'])
    .groupby('byggeaar', as_index=False)['antall_boliger'].sum()
    .sort_values('byggeaar')
    .reset_index(drop=True)
)
bygningsbestand_2022


/Users/marcus/repositories/Beregninger_Scenariomodell/.venv/lib/python3.14/site-packages/pandas/core/frame.py:5246: ChainedAssignmentError: A value is trying to be set on a copy of a DataFrame or Series through chained assignment.
When using the Copy-on-Write mode, such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy.

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[k] = com.apply_if_callable(v, data)


,byggeaar,antall_boliger
0,1900,79505.0
1,1901,2407.0
2,1902,2407.0
3,1903,2407.0
4,1904,2407.0
...,...,...
118,2018,7683.3
119,2019,7683.3
120,2020,7683.3
121,2021,3052.0


https://www.ssb.no/statbank/table/10573

scaling dactor for bolig to enebolig  = 22 848 / 17 518	

In [ ]:
# Hent energiforbruk per husholdning fra SSB 10576
meta_energy = requests.get("https://data.ssb.no/api/pxwebapi/v2/tables/10576/metadata?lang=no")
energy_labels = meta_energy.json()['dimension']['Energibaerer']['category']['label']

r = requests.get("https://data.ssb.no/api/pxwebapi/v2/tables/10576/data", params={
    'lang': 'no',
    'outputformat': 'csv',
    'valueCodes[Energibaerer]': '*',
    'valueCodes[ByggeAar]': '*',
    'valueCodes[ContentsCode]': 'Forbruk',
    'valueCodes[Tid]': '2022',
})
df_energy = pd.read_csv(StringIO(r.text)).rename(columns={'Forbruk 2022': 'Forbruk'})
df_energy.loc[:, 'Forbruk'] = pd.to_numeric(df_energy['Forbruk'], errors='coerce')
df_energy


In [5]:

# Definer årsintervaller for 10576 byggeår-koder
energy_year_ranges = {
    30: (1900, 1930),   # Før 1931
    31: (1931, 1954),
    51: (1955, 1970),
    71: (1971, 1986),
    81: (1987, 1996),
    91: (1997, 2009),
    92: (2010, 2022),
}

expand_energy = pd.DataFrame([
    {'ByggeAar': code, 'byggeaar': y}
    for code, (start, end) in energy_year_ranges.items()
    for y in range(start, end + 1)
])

df_energy_wide = (
    df_energy
    .pivot(index='ByggeAar', columns='Energibaerer', values='Forbruk')
    .rename(columns=energy_labels)
    .reset_index()
)

gjennomsnittlig_forbruk_2022 = (
    df_energy_wide
    .merge(expand_energy, on='ByggeAar')
    .drop(columns=['ByggeAar', 'Olje og parafin', 'I alt'])
    .sort_values('byggeaar')
    .reset_index(drop=True)
)
gjennomsnittlig_forbruk_2022


,Elektrisitet,"Ved, pellets og vedbriketter",Gass og fjernvarme,byggeaar
0,14385.0,2627.0,224.0,1900
1,14385.0,2627.0,224.0,1901
2,14385.0,2627.0,224.0,1902
3,14385.0,2627.0,224.0,1903
4,14385.0,2627.0,224.0,1904
...,...,...,...,...
118,12559.0,882.0,948.0,2018
119,12559.0,882.0,948.0,2019
120,12559.0,882.0,948.0,2020
121,12559.0,882.0,948.0,2021


In [6]:
meta = requests.get("https://data.ssb.no/api/pxwebapi/v2/tables/06265/metadata?lang=no")
meta.json()

{'version': '2.0',
 'class': 'dataset',
 'label': '06265: Boliger, etter bygningstype (K) 2006-2025',
 'source': 'Statistisk sentralbyrå',
 'updated': '2025-03-04T07:00:00Z',
 'note': ['Populasjon: \nNye tall for  boligpopulasjon 2012, 2013, 2014, 2015 og 2016. Modell for avgrensing av boligpopulasjonen er noe endret, slik at det blir et brudd i statistikken før og etter 2012. Boligpopulasjonen avgrenses vha Matrikkelen og Folkeregisteret. Kjennemerkene som brukes er bygningstype, bygningsstatus, bruksenhetstype og hvorvidt det er registrert bosatte i bruksenhet eller bygning. Fritidsboliger og bruksenheter registrert som annet enn bolig inkluderes i statistikken dersom det ifølge folkeregisteret er registrert bosatte på boligens adresse.  Folkeregisteret brukes også til å overstyre informasjon, eksempelvis ved manglende ferdigattest, dersom det er registrert bosatte i bygningen. I tillegg imputeres boliger i boligbygninger med registrert bosatte men uten registrerte bruksenheter.',
  

In [7]:
meta = requests.get("https://data.ssb.no/api/pxwebapi/v2/tables/14282/metadata?lang=no")
meta.json()

{'version': '2.0',
 'class': 'dataset',
 'label': '14282: Framskrevet folkemengde 1. januar, etter kjønn, alder, innvandringskategori og landbakgrunn, i 15 alternativer 2024-2100',
 'source': 'Statistisk sentralbyrå',
 'updated': '2024-06-05T06:00:00Z',
 'note': ['Hvert alternativ beskrives ved tre bokstaver i følgende rekkefølge: fruktbarhet, levealder og innvandring. M = middels, L = lav, H = høy, K = konstant, E = ingen nettoinnvandring og 0 = ingen flytting.'],
 'role': {'time': ['Tid'], 'metric': ['ContentsCode']},
 'id': ['Kjonn',
  'Alder',
  'InnvandrLandbakgr',
  'Framskriv',
  'ContentsCode',
  'Tid'],
 'size': [2, 106, 8, 15, 1, 77],
 'dimension': {'Kjonn': {'label': 'kjønn',
   'category': {'index': {'2': 0, '1': 1},
    'label': {'2': 'Kvinner', '1': 'Menn'}},
   'extension': {'elimination': True, 'show': 'value', 'codelists': []},
   'link': {'describedby': [{'extension': {'Kjonn': 'urn:ssb:classification:klass:2'}}]}},
  'Alder': {'label': 'alder',
   'category': {'index

### LLM overviw of relevant SSB data

**Tables you already use:**

| Tabell | Innhold | Bruk |
|---|---|---|
| **06266** | Boliger etter bygningstype og byggeår (K), 2006–2025 | Boligbestand per byggeår-kohort |
| **10576** | Gj.snittlig energiforbruk per husholdning, etter energibærer og byggeår, 2009/2012/2022 | kWh per bolig per kohort |

---

**Tables you should grab for the projection:**

| Tabell | Innhold | Hva du bruker den til |
|---|---|---|
| **06265** | Boliger etter bygningstype (K), 2006–2025 | Total boligbestand over tid (enklere tidsserie enn 06266 for å se netto vekst/år) |
| **05940** | Byggeareal: fullførte boliger + bruksareal, etter bygningstype (K), 2000–2024 (årlig) | Historisk nybygging av eneboliger per år → grunnlag for fremtidig nybyggtakt |
| **10784** | Byggeareal: avgang av boliger, etter bygningstype (K), 2009K1–2024 (kvartalsvis) | Historisk rivning/avgang per år → grunnlag for rivningsrate |
| **10785** | Byggeareal: avgang av bygninger, etter bygningstype (K), 2009–2021 (årlig) | Samme som over men på bygningsnivå, årlig — enklere å bruke |
| **13929** | Energiforbruk i husholdninger og fritidshus, 1990–2024 | Total energibruk i husholdninger over tid (GWh) — god for sanity check og for å se den historiske trenden i total energibruk |
| **10573** | Gj.snittlig energiforbruk per husholdning, etter hustype, 1995–2022 | Energitrend spesifikt for eneboliger over tid (viser den naturlige nedgangen uten spesifikk etterisolering) |
| **10582** | Gj.snittlig energiforbruk per husholdning, etter energibærer og hustype, 1995–2022 | Samme som over men splittet på energibærer — viser f.eks. at olje forsvinner over tid |

---

**Nice to have (men ikke strengt nødvendig):**

| Tabell | Innhold | Eventuell bruk |
|---|---|---|
| **06513** | Boliger etter bygningstype og bruksareal (K), 2007–2025 | Areal per bolig per kohort — hvis du vil gå fra kWh/m² i stedet |
| **10571** | Husholdninger etter type hovedoppvarmingskilde (%), 1994–2022 | Viser varmepumpe-utbredelse over tid |
| **14578** | Energiforbruk per husholdning, etter energibærer, landsdel og fylke, 2022 | Regional variasjon (om relevant) |
| **13932** | Klimagasser fra norsk øk. aktivitet, etter næring (AR5/Paris) | For å kryssjekke dine utslippstall mot offisielle husholdningsutslipp |

---

De fire øverste i "bør hente"-lista er det viktigste: **05940** og **10784/10785** gir deg nybygg- og rivningsrater, **13929** gir en sanity-check på totalt energiforbruk, og **10573** viser den historiske trenden i energibruk per enebolig som du kan bruke til å kalibrere din "naturlig forbedring"-antakelse i baseline.

In [8]:

# SSB 10576 gir snitt for alle boliger, ikke eneboliger.
# Korreksjonsfaktor fra SSB 10573: enebolig / alle boliger
enebolig_faktor = 22_848 / 17_518

all_years = pd.DataFrame({'byggeaar': np.arange(1900, 2101)})

boligbestand_profil = (
    all_years
    .merge(bygningsbestand_2022.rename(columns={'antall_boliger': 'antall'}), on='byggeaar', how='left')
    .merge(gjennomsnittlig_forbruk_2022.rename(columns={
        'Elektrisitet':                  'elektrisitet',
        'Ved, pellets og vedbriketter':  'ved',
        'Gass og fjernvarme':            'fjernvarme',
    }), on='byggeaar', how='left')
    .fillna(0)
    .assign(
        energi=lambda d: (d['elektrisitet'] + d['ved'] + d['fjernvarme']) * enebolig_faktor,
        varmepumpe=0.0,
        etterisolert=0.0,
    )
    [['byggeaar', 'antall', 'varmepumpe', 'etterisolert', 'energi']]
)

boligbestand_profil.to_csv('data/bygningsbestand_2022.csv', index=False)
boligbestand_profil


/Users/marcus/repositories/Beregninger_Scenariomodell/.venv/lib/python3.14/site-packages/pandas/core/frame.py:5246: ChainedAssignmentError: A value is trying to be set on a copy of a DataFrame or Series through chained assignment.
When using the Copy-on-Write mode, such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy.

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[k] = com.apply_if_callable(v, data)
/Users/marcus/repositories/Beregninger_Scenariomodell/.venv/lib/python3.14/site-packages/pandas/core/frame.py:5246: ChainedAssignmentError: A value is trying to be set on a copy of a DataFrame or Series through chained assignment.
When using the Copy-on-Write mode, such chaine

,byggeaar,antall,varmepumpe,etterisolert,energi
0,1900,79505.0,0.0,0.0,22480.199109
1,1901,2407.0,0.0,0.0,22480.199109
2,1902,2407.0,0.0,0.0,22480.199109
3,1903,2407.0,0.0,0.0,22480.199109
4,1904,2407.0,0.0,0.0,22480.199109
...,...,...,...,...,...
196,2096,0.0,0.0,0.0,0.000000
197,2097,0.0,0.0,0.0,0.000000
198,2098,0.0,0.0,0.0,0.000000
199,2099,0.0,0.0,0.0,0.000000
